In [3]:
# Colab / Deepnote / local GPU notebook
# Attack-only pipeline: train a lightweight differentiable detector (CLIP + linear head),
# then run PGD and EOT-PGD (transform-aware) attacks and report attack success + AUROC drop.

!pip -q install -U torch torchvision torchaudio
!pip -q install -U transformers datasets accelerate evaluate scikit-learn tqdm
!pip -q install -U kornia
!pip -q install -U diffjpeg  # differentiable JPEG (for EOT); if it fails, code falls back gracefully


[notice] A new release of pip is available: 25.3 -> 26.0.1
[notice] To update, run: pip install --upgrade pip

[notice] A new release of pip is available: 25.3 -> 26.0.1
[notice] To update, run: pip install --upgrade pip

[notice] A new release of pip is available: 25.3 -> 26.0.1
[notice] To update, run: pip install --upgrade pip
ERROR: Could not find a version that satisfies the requirement diffjpeg (from versions: none)

[notice] A new release of pip is available: 25.3 -> 26.0.1
[notice] To update, run: pip install --upgrade pip
ERROR: No matching distribution found for diffjpeg


In [10]:
import os, random, math
import numpy as np
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import DataLoader

from datasets import load_dataset
from transformers import CLIPImageProcessor, CLIPVisionModel
from sklearn.metrics import roc_auc_score, accuracy_score
from tqdm.auto import tqdm

import kornia.augmentation as K

device = "cuda" if torch.cuda.is_available() else "cpu"
torch.manual_seed(0); np.random.seed(0); random.seed(0)
print("device:", device)

AttributeError: module 'torch.utils._pytree' has no attribute 'PyTreeSpec'

In [12]:
import torch
import torchvision
from torchvision import transforms
from torch.utils.data import DataLoader

# Transform (CLIP-compatible size later)
transform = transforms.Compose([
    transforms.Resize((224, 224)),
    transforms.ToTensor(),
])

# STL10 Real Images
real_dataset = torchvision.datasets.STL10(
    root="./data",
    split="train",
    download=True,
    transform=transform
)

print("Real dataset size:", len(real_dataset))

100%|██████████| 2.64G/2.64G [01:51<00:00, 23.6MB/s]
Real dataset size: 5000


In [18]:
!pip -q install diffusers transformers accelerate safetensors

import torch
from diffusers import StableDiffusionPipeline
from tqdm import tqdm
import os
import random

device = "cuda" if torch.cuda.is_available() else "cpu"

# Load SD 1.5 (lighter than SDXL, good for T4)
pipe = StableDiffusionPipeline.from_pretrained(
    "runwayml/stable-diffusion-v1-5",
    torch_dtype=torch.float16
).to(device)

pipe.enable_attention_slicing()
pipe.enable_vae_slicing()

os.makedirs("fake_images", exist_ok=True)

# -------------------------
# 10 diverse prompts
# -------------------------
prompts = [
    "a realistic DSLR photo of a busy city street",
    "a professional wildlife photograph of a tiger in forest",
    "a candid iphone photo of friends at a cafe",
    "a high resolution landscape photo of mountains at sunset",
    "a news style photo of a political rally",
    "a sports photography shot of a soccer match",
    "a portrait photo with natural lighting and shallow depth of field",
    "a travel photography shot of a beach with tourists",
    "a nighttime street photography image with neon lights",
    "a documentary style photo of a protest scene"
]

images_per_prompt = 100

print("Starting generation...")

for prompt_id, prompt in enumerate(prompts):
    print(f"\nGenerating for prompt {prompt_id+1}/10")
    
    for i in tqdm(range(images_per_prompt)):
        seed = random.randint(0, 1_000_000)
        generator = torch.Generator(device=device).manual_seed(seed)
        
        image = pipe(
            prompt,
            guidance_scale=7.5,
            num_inference_steps=20,
            generator=generator
        ).images[0]
        
        image.save(f"fake_images/p{prompt_id}_{i}.png")

print("\nAll fake images generated ✅")

100%|██████████| 100/100 [08:04<00:00,  4.84s/it]
All fake images generated ✅



In [6]:
import os, glob, random
from PIL import Image

import torch
from torch.utils.data import Dataset, DataLoader, random_split
import torchvision
from torchvision import transforms

device = "cuda" if torch.cuda.is_available() else "cpu"
print("device:", device)

# --- settings ---
FAKE_DIR = "fake_images"      # where your SD images are saved
N_FAKE_MAX = 1000             # 10 prompts x 100 each
N_REAL_MAX = 1000             # match fake count for balance

IMG_SIZE = 224
BATCH_SIZE = 64

# Real images (STL10)
real_transform = transforms.Compose([
    transforms.Resize((IMG_SIZE, IMG_SIZE)),
    transforms.ToTensor(),   # -> [0,1]
])

real_ds_full = torchvision.datasets.STL10(
    root="./data", split="train", download=True, transform=real_transform
)

# Subsample real for balance
indices = list(range(len(real_ds_full)))
random.shuffle(indices)
indices = indices[:N_REAL_MAX]
real_ds = torch.utils.data.Subset(real_ds_full, indices)

# Fake images from disk
fake_paths = sorted(glob.glob(os.path.join(FAKE_DIR, "*.png")))[:N_FAKE_MAX]
assert len(fake_paths) > 0, f"No fake images found in {FAKE_DIR}"

fake_transform = transforms.Compose([
    transforms.Resize((IMG_SIZE, IMG_SIZE)),
    transforms.ToTensor(),   # -> [0,1]
])

class FakeFolderDataset(Dataset):
    def __init__(self, paths, transform):
        self.paths = paths
        self.transform = transform

    def __len__(self): return len(self.paths)

    def __getitem__(self, idx):
        p = self.paths[idx]
        img = Image.open(p).convert("RGB")
        x = self.transform(img)
        y = 1  # fake
        return x, torch.tensor(y, dtype=torch.long)

fake_ds = FakeFolderDataset(fake_paths, fake_transform)

print("Real:", len(real_ds), "Fake:", len(fake_ds))

device: cuda
Real: 1000 Fake: 1000


In [7]:
from torch.utils.data import ConcatDataset

class WithLabel(Dataset):
    def __init__(self, base, label):
        self.base = base
        self.label = label
    def __len__(self): return len(self.base)
    def __getitem__(self, idx):
        x, _ = self.base[idx]  # STL10 returns (img, class). We ignore class.
        return x, torch.tensor(self.label, dtype=torch.long)

real_labeled = WithLabel(real_ds, 0)  # 0 = real

full_ds = ConcatDataset([real_labeled, fake_ds])

# Split
train_frac = 0.8
n_train = int(len(full_ds) * train_frac)
n_test = len(full_ds) - n_train
train_ds, test_ds = random_split(full_ds, [n_train, n_test], generator=torch.Generator().manual_seed(0))

train_loader = DataLoader(train_ds, batch_size=BATCH_SIZE, shuffle=True, num_workers=2, pin_memory=True)
test_loader  = DataLoader(test_ds,  batch_size=BATCH_SIZE, shuffle=False, num_workers=2, pin_memory=True)

print("Train:", len(train_ds), "Test:", len(test_ds))

Train: 1600 Test: 400


In [27]:
!pip -q install -U transformers

import torch.nn as nn
import torch.nn.functional as F
from transformers import CLIPVisionModel

class ClipDetector(nn.Module):
    def __init__(self):
        super().__init__()
        self.backbone = CLIPVisionModel.from_pretrained("openai/clip-vit-base-patch32")
        hidden = self.backbone.config.hidden_size
        self.head = nn.Linear(hidden, 2)

    def forward(self, x01):
        # x01: [0,1] tensor (B,3,224,224)
        # CLIP expects normalized inputs; do it here for correct gradients
        mean = torch.tensor([0.48145466, 0.4578275, 0.40821073], device=x01.device).view(1,3,1,1)
        std  = torch.tensor([0.26862954, 0.26130258, 0.27577711], device=x01.device).view(1,3,1,1)
        x = (x01 - mean) / std

        feats = self.backbone(pixel_values=x).pooler_output
        return self.head(feats)

model = ClipDetector().to(device)

# freeze backbone for speed (still differentiable w.r.t. input)
for p in model.backbone.parameters():
    p.requires_grad = False

opt = torch.optim.AdamW(model.head.parameters(), lr=3e-4)


[notice] A new release of pip is available: 25.3 -> 26.0.1
[notice] To update, run: pip install --upgrade pip
Loading weights: 100%|██████████| 199/199 [00:01<00:00, 126.31it/s, Materializing param=vision_model.pre_layrnorm.weight]
CLIPVisionModel LOAD REPORT from: openai/clip-vit-base-patch32
Key                                                          | Status     |  | 
-------------------------------------------------------------+------------+--+-
text_model.encoder.layers.{0...11}.mlp.fc1.weight            | UNEXPECTED |  | 
text_model.encoder.layers.{0...11}.self_attn.v_proj.weight   | UNEXPECTED |  | 
text_model.encoder.layers.{0...11}.self_attn.out_proj.bias   | UNEXPECTED |  | 
text_model.encoder.layers.{0...11}.self_attn.k_proj.bias     | UNEXPECTED |  | 
text_model.encoder.layers.{0...11}.layer_norm1.bias          | UNEXPECTED |  | 
text_model.encoder.layers.{0...11}.self_attn.q_proj.weight   | UNEXPECTED |  | 
text_model.final_layer_norm.bias                             | U

In [30]:
from tqdm.auto import tqdm
import numpy as np
from sklearn.metrics import roc_auc_score, accuracy_score

def train_one_epoch():
    model.train()
    tot = 0.0
    for x, y in tqdm(train_loader, desc="train", leave=False):
        x = x.to(device); y = y.to(device)
        opt.zero_grad(set_to_none=True)
        logits = model(x)
        loss = F.cross_entropy(logits, y)
        loss.backward()
        opt.step()
        tot += loss.item() * x.size(0)
    return tot / len(train_loader.dataset)

@torch.no_grad()
def eval_clean():
    model.eval()
    ys, ps = [], []
    for x, y in tqdm(test_loader, desc="eval", leave=False):
        x = x.to(device)
        prob_fake = torch.softmax(model(x), dim=-1)[:,1].cpu().numpy()
        ys.append(y.numpy()); ps.append(prob_fake)
    y = np.concatenate(ys); p = np.concatenate(ps)
    return {
        "auc": float(roc_auc_score(y, p)),
        "acc@0.5": float(accuracy_score(y, (p >= 0.5).astype(int)))
    }

loss = train_one_epoch()   # 1 epoch is enough for a usable baseline
clean = eval_clean()
print("train loss:", loss)
print("CLEAN metrics:", clean)

train loss: 0.45045307219028474
CLEAN metrics: {'auc': 0.996525, 'acc@0.5': 0.98}


In [33]:
!pip -q install -U kornia
import kornia.augmentation as K

# Optional differentiable JPEG; if install fails, EOT still works with geo+color+blur
try:
    !pip -q install -U diffjpeg
    from diffjpeg import DiffJPEG
    HAS_JPEG = True
except Exception:
    HAS_JPEG = False

class FBLikeTransforms(nn.Module):
    def __init__(self, h=224, w=224):
        super().__init__()
        self.geom = nn.Sequential(
            K.RandomResizedCrop((h,w), scale=(0.85, 1.0), ratio=(0.9, 1.1), p=1.0),
        )
        self.color = nn.Sequential(
            K.RandomGamma(gamma=(0.9, 1.1), p=0.7),
            K.RandomBrightness(brightness=(0.9, 1.1), p=0.7),
            K.RandomContrast(contrast=(0.9, 1.1), p=0.7),
        )
        self.blur = K.RandomGaussianBlur((3,3), sigma=(0.1, 1.2), p=0.3)
        self.jpeg = DiffJPEG(height=h, width=w, differentiable=True, quality=85) if HAS_JPEG else None

    def forward(self, x01):
        x = self.geom(x01)
        x = self.color(x)
        x = self.blur(x)
        if self.jpeg is not None:
            x = self.jpeg(x.clamp(0,1))
        return x.clamp(0,1)

T = FBLikeTransforms().to(device)

def meme_band_mask(x, band_frac=0.22, where="bottom"):
    B,C,H,W = x.shape
    mask = torch.zeros_like(x)
    h = int(H * band_frac)
    if where == "bottom":
        mask[:,:,H-h:H,:] = 1.0
    elif where == "top":
        mask[:,:,:h,:] = 1.0
    else:
        raise ValueError("where must be top/bottom")
    return mask

def eot_pgd_targeted_masked(model, x01, y_target, T, eps=8/255, alpha=2/255, steps=30, eot_k=4, mask=None):
    """
    Attack only on masked region. Target label is 'real'(0) for fake samples.
    x01 in [0,1]
    """
    model.eval()
    x0 = x01.detach()
    if mask is None:
        mask = torch.ones_like(x0)

    delta = torch.zeros_like(x0).uniform_(-eps, eps).to(x0.device)
    delta = (delta * mask).requires_grad_(True)

    for _ in range(steps):
        loss = 0.0
        for _k in range(eot_k):
            adv = (x0 + delta).clamp(0,1)
            adv_t = T(adv)
            logits = model(adv_t)
            loss = loss + F.cross_entropy(logits, y_target)
        loss = loss / eot_k

        grad = torch.autograd.grad(loss, delta, retain_graph=False, create_graph=False)[0]
        delta.data = (delta.data - alpha * grad.sign()) * mask
        delta.data = delta.data.clamp(-eps, eps)
        delta.grad = None

    return (x0 + delta.detach()).clamp(0,1)


[notice] A new release of pip is available: 25.3 -> 26.0.1
[notice] To update, run: pip install --upgrade pip
ERROR: Could not find a version that satisfies the requirement diffjpeg (from versions: none)

[notice] A new release of pip is available: 25.3 -> 26.0.1
[notice] To update, run: pip install --upgrade pip
ERROR: No matching distribution found for diffjpeg


In [36]:
@torch.no_grad()
def prob_fake(x01):
    return torch.softmax(model(x01), dim=-1)[:,1]

def eval_attack(mask_where="bottom", max_batches=20):
    model.eval()
    ys_all, ps_all = [], []
    succ, total = 0, 0

    for bi, (x, y) in enumerate(tqdm(test_loader, desc="attack-eval")):
        if bi >= max_batches:
            break
        x = x.to(device); y = y.to(device)
        x_adv = x.clone()

        fake_mask = (y == 1)
        if fake_mask.any():
            xf = x[fake_mask]
            yt = torch.zeros(xf.size(0), dtype=torch.long, device=device)  # target real
            mask = meme_band_mask(xf, band_frac=0.22, where=mask_where)
            xf_adv = eot_pgd_targeted_masked(model, xf, yt, T, eps=8/255, alpha=2/255, steps=30, eot_k=4, mask=mask)
            x_adv[fake_mask] = xf_adv

            pf = prob_fake(x_adv)[y == 1]
            succ += (pf < 0.5).sum().item()
            total += pf.numel()

        p = prob_fake(x_adv).detach().cpu().numpy()
        ys_all.append(y.detach().cpu().numpy())
        ps_all.append(p)

    y_all = np.concatenate(ys_all)
    p_all = np.concatenate(ps_all)
    auc = roc_auc_score(y_all, p_all)
    return {
        "attack_success_fake_to_real@0.5": float(succ / max(1, total)),
        "auc_after_attack": float(auc),
        "n_fakes_attacked": int(total),
        "mask_where": mask_where
    }

print("CLEAN:", eval_clean())
print("ATTACK (bottom band):", eval_attack("bottom", max_batches=20))
print("ATTACK (top band):", eval_attack("top", max_batches=20))

CLEAN: {'auc': 0.996525, 'acc@0.5': 0.98}
attack-eval: 100%|██████████| 7/7 [02:59<00:00, 25.70s/it]
ATTACK (bottom band): {'attack_success_fake_to_real@0.5': 0.78, 'auc_after_attack': 0.70465, 'n_fakes_attacked': 200, 'mask_where': 'bottom'}
attack-eval: 100%|██████████| 7/7 [02:58<00:00, 25.56s/it]ATTACK (top band): {'attack_success_fake_to_real@0.5': 0.76, 'auc_after_attack': 0.73305, 'n_fakes_attacked': 200, 'mask_where': 'top'}



In [39]:
# ============================================================
# (A) Rebuild TEST loader WITH METADATA (prompt_id, path, label)
# Run this AFTER your training/attack code. It will NOT retrain.
# ============================================================
import os, glob, random, re
from PIL import Image

import torch
from torch.utils.data import Dataset, DataLoader, random_split
import torchvision
from torchvision import transforms

device = "cuda" if torch.cuda.is_available() else "cpu"

FAKE_DIR = "fake_images"
N_FAKE_MAX = 1000
N_REAL_MAX = 1000
IMG_SIZE = 224
BATCH_SIZE = 64

real_transform = transforms.Compose([
    transforms.Resize((IMG_SIZE, IMG_SIZE)),
    transforms.ToTensor(),  # [0,1]
])

fake_transform = transforms.Compose([
    transforms.Resize((IMG_SIZE, IMG_SIZE)),
    transforms.ToTensor(),  # [0,1]
])

# ---- Real dataset (STL10)
real_ds_full = torchvision.datasets.STL10(
    root="./data", split="train", download=True, transform=real_transform
)

# deterministically subsample real (same idea as before, but stable)
g = torch.Generator().manual_seed(0)
perm = torch.randperm(len(real_ds_full), generator=g).tolist()
real_indices = perm[:N_REAL_MAX]
real_subset = torch.utils.data.Subset(real_ds_full, real_indices)

class RealWithMeta(Dataset):
    def __init__(self, subset):
        self.subset = subset
    def __len__(self): return len(self.subset)
    def __getitem__(self, idx):
        x, _ = self.subset[idx]
        return {
            "x": x,
            "y": torch.tensor(0, dtype=torch.long),
            "path": None,
            "prompt_id": -1,
            "is_fake": False,
        }

# ---- Fake dataset from disk with prompt_id parsed from filename p{prompt}_{i}.png
fake_paths = sorted(glob.glob(os.path.join(FAKE_DIR, "*.png")))[:N_FAKE_MAX]
assert len(fake_paths) > 0, f"No fake images found in {FAKE_DIR}"

_prompt_re = re.compile(r".*[/\\]p(\d+)_\d+\.png$")

class FakeFolderWithMeta(Dataset):
    def __init__(self, paths, transform):
        self.paths = paths
        self.transform = transform
    def __len__(self): return len(self.paths)
    def __getitem__(self, idx):
        p = self.paths[idx]
        img = Image.open(p).convert("RGB")
        x = self.transform(img)
        m = _prompt_re.match(p)
        prompt_id = int(m.group(1)) if m else -1
        return {
            "x": x,
            "y": torch.tensor(1, dtype=torch.long),
            "path": p,
            "prompt_id": prompt_id,
            "is_fake": True,
        }

real_meta = RealWithMeta(real_subset)
fake_meta = FakeFolderWithMeta(fake_paths, fake_transform)

class ConcatMeta(Dataset):
    def __init__(self, a, b):
        self.a, self.b = a, b
    def __len__(self): return len(self.a) + len(self.b)
    def __getitem__(self, idx):
        if idx < len(self.a):
            return self.a[idx]
        return self.b[idx - len(self.a)]

full_meta = ConcatMeta(real_meta, fake_meta)

# Split (match your earlier split seed)
train_frac = 0.8
n_train = int(len(full_meta) * train_frac)
n_test = len(full_meta) - n_train
train_meta, test_meta = random_split(full_meta, [n_train, n_test], generator=torch.Generator().manual_seed(0))

def collate_meta(batch):
    x = torch.stack([b["x"] for b in batch], dim=0)
    y = torch.stack([b["y"] for b in batch], dim=0)
    prompt_id = torch.tensor([b["prompt_id"] for b in batch], dtype=torch.long)
    is_fake = torch.tensor([int(b["is_fake"]) for b in batch], dtype=torch.long)
    paths = [b["path"] for b in batch]
    return x, y, prompt_id, is_fake, paths

test_loader_meta = DataLoader(
    test_meta, batch_size=BATCH_SIZE, shuffle=False, num_workers=2, pin_memory=True, collate_fn=collate_meta
)

print("Rebuilt meta test set:", len(test_meta))
print("Fake in meta test set:", sum(int(full_meta[i]["is_fake"]) for i in test_meta.indices))

Rebuilt meta test set: 400
Fake in meta test set: 200


In [42]:
# ============================================================
# (B) Utilities: ECE + reliability curve bins + plots
# Saves CSVs and PNGs under ./paper_outputs
# ============================================================
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from sklearn.metrics import roc_auc_score, accuracy_score

OUTDIR = "paper_outputs"
os.makedirs(OUTDIR, exist_ok=True)

@torch.no_grad()
def prob_fake01(x01):
    model.eval()
    return torch.softmax(model(x01), dim=-1)[:, 1]

def ece_and_bins(y_true, p_fake, n_bins=15):
    """
    y_true: 0/1
    p_fake: predicted probability of fake (float in [0,1])
    Returns: ece (float), bins_df (pd.DataFrame)
    """
    y_true = np.asarray(y_true).astype(int)
    p = np.asarray(p_fake).astype(float)
    bins = np.linspace(0.0, 1.0, n_bins + 1)
    bin_ids = np.digitize(p, bins) - 1
    bin_ids = np.clip(bin_ids, 0, n_bins - 1)

    rows = []
    ece = 0.0
    N = len(p)
    for b in range(n_bins):
        mask = (bin_ids == b)
        if mask.sum() == 0:
            rows.append([b, bins[b], bins[b+1], 0, np.nan, np.nan, np.nan])
            continue
        conf = p[mask].mean()
        acc = y_true[mask].mean()   # fraction of fakes in that bin
        cnt = mask.sum()
        ece += (cnt / N) * abs(acc - conf)
        rows.append([b, bins[b], bins[b+1], cnt, conf, acc, abs(acc - conf)])
    df = pd.DataFrame(rows, columns=["bin", "lo", "hi", "count", "avg_conf", "avg_acc", "abs_gap"])
    return float(ece), df

def plot_reliability(df, title, path_png):
    # Plot avg_acc vs avg_conf; only bins with count>0
    d = df[df["count"] > 0].copy()
    plt.figure()
    plt.plot([0,1], [0,1])
    plt.plot(d["avg_conf"].values, d["avg_acc"].values, marker="o")
    plt.xlabel("Confidence (P(fake))")
    plt.ylabel("Empirical accuracy (fraction fake)")
    plt.title(title)
    plt.savefig(path_png, dpi=200, bbox_inches="tight")
    plt.close()

def plot_conf_hist(p_before, p_after, title, path_png):
    plt.figure()
    plt.hist(p_before, bins=30, alpha=0.6, label="before")
    plt.hist(p_after, bins=30, alpha=0.6, label="after")
    plt.xlabel("P(fake)")
    plt.ylabel("Count")
    plt.title(title)
    plt.legend()
    plt.savefig(path_png, dpi=200, bbox_inches="tight")
    plt.close()

print("Outputs will be saved to:", OUTDIR)

Outputs will be saved to: paper_outputs


In [45]:
# ============================================================
# (C) Run CLEAN vs ATTACK eval with:
#     - overall metrics
#     - per-prompt breakdown
#     - calibration curves (ECE + reliability diagram)
#     - confidence histograms
# Saves:
#   - overall_metrics.csv
#   - per_prompt_metrics.csv
#   - per_sample_outputs.csv
#   - reliability_clean.csv/.png
#   - reliability_attack_*.csv/.png
#   - conf_hist_fakes_*.png
# ============================================================
from tqdm.auto import tqdm

# Use your existing T and eot_pgd_targeted_masked + meme_band_mask
# Assumes x are in [0,1]

def run_eval(mode="clean", mask_where="bottom", max_batches=None,
             eps=8/255, alpha=2/255, steps=30, eot_k=4):
    """
    mode: "clean" or "attack"
    Returns per-sample dataframe + overall dict + per-prompt dataframe
    """
    model.eval()
    rows = []

    for bi, (x, y, prompt_id, is_fake, paths) in enumerate(tqdm(test_loader_meta, desc=f"eval-{mode}")):
        if (max_batches is not None) and (bi >= max_batches):
            break
        x = x.to(device)
        y = y.to(device)
        prompt_id = prompt_id.to(device)
        is_fake = is_fake.to(device)

        # predictions before
        p0 = prob_fake01(x).detach()

        x_use = x
        if mode == "attack":
            # attack only fakes in the batch
            mask = (y == 1)
            if mask.any():
                xf = x[mask]
                yt = torch.zeros(xf.size(0), dtype=torch.long, device=device)  # target real
                m = meme_band_mask(xf, band_frac=0.22, where=mask_where)
                xf_adv = eot_pgd_targeted_masked(
                    model, xf, yt, T,
                    eps=eps, alpha=alpha, steps=steps, eot_k=eot_k, mask=m
                )
                x_adv = x.clone()
                x_adv[mask] = xf_adv
                x_use = x_adv

        # predictions after (or clean)
        p1 = prob_fake01(x_use).detach()

        # record per sample
        for i in range(x.size(0)):
            rows.append({
                "y": int(y[i].item()),
                "is_fake": int(is_fake[i].item()),
                "prompt_id": int(prompt_id[i].item()),
                "path": paths[i],
                "p_fake_before": float(p0[i].item()),
                "p_fake_after": float(p1[i].item()),
                "mode": mode,
                "mask_where": mask_where if mode == "attack" else "",
            })

    df = pd.DataFrame(rows)

    # overall metrics computed using p_fake_after (for clean, before==after but we use after)
    y_true = df["y"].values
    p = df["p_fake_after"].values
    auc = roc_auc_score(y_true, p)
    acc = accuracy_score(y_true, (p >= 0.5).astype(int))

    # attack success on fakes: fake -> predicted real
    df_f = df[df["y"] == 1].copy()
    attack_succ = float(((df_f["p_fake_after"].values < 0.5).astype(int)).mean()) if len(df_f) else np.nan

    overall = {
        "mode": mode,
        "mask_where": mask_where if mode == "attack" else "",
        "n_samples": int(len(df)),
        "n_fakes": int((df["y"] == 1).sum()),
        "auc": float(auc),
        "acc@0.5": float(acc),
        "attack_success_fake_to_real@0.5": float(attack_succ) if mode == "attack" else np.nan,
        "eps": float(eps) if mode == "attack" else np.nan,
        "steps": int(steps) if mode == "attack" else np.nan,
        "eot_k": int(eot_k) if mode == "attack" else np.nan,
    }

    # per-prompt breakdown (only for fakes; prompt_id meaningful there)
    per_prompt = []
    for pid, g in df[df["y"] == 1].groupby("prompt_id"):
        per_prompt.append({
            "prompt_id": int(pid),
            "n": int(len(g)),
            "attack_success@0.5": float((g["p_fake_after"].values < 0.5).mean()) if mode == "attack" else np.nan,
            "mean_p_fake_before": float(g["p_fake_before"].mean()),
            "mean_p_fake_after": float(g["p_fake_after"].mean()),
            "median_p_fake_before": float(g["p_fake_before"].median()),
            "median_p_fake_after": float(g["p_fake_after"].median()),
        })
    per_prompt_df = pd.DataFrame(per_prompt).sort_values("prompt_id")

    return df, overall, per_prompt_df

# ---- Run clean
df_clean, overall_clean, per_prompt_clean = run_eval(mode="clean")

# ---- Run attacks (bottom + top)
df_attack_bottom, overall_bottom, per_prompt_bottom = run_eval(
    mode="attack", mask_where="bottom", eps=8/255, alpha=2/255, steps=30, eot_k=4
)
df_attack_top, overall_top, per_prompt_top = run_eval(
    mode="attack", mask_where="top", eps=8/255, alpha=2/255, steps=30, eot_k=4
)

# ---- Save CSVs
overall_df = pd.DataFrame([overall_clean, overall_bottom, overall_top])
overall_df.to_csv(os.path.join(OUTDIR, "overall_metrics.csv"), index=False)

per_prompt_bottom.to_csv(os.path.join(OUTDIR, "per_prompt_metrics_attack_bottom.csv"), index=False)
per_prompt_top.to_csv(os.path.join(OUTDIR, "per_prompt_metrics_attack_top.csv"), index=False)
per_prompt_clean.to_csv(os.path.join(OUTDIR, "per_prompt_metrics_clean.csv"), index=False)

# Per-sample (can be big; still fine for ~400-500 samples)
pd.concat([df_clean, df_attack_bottom, df_attack_top], ignore_index=True)\
  .to_csv(os.path.join(OUTDIR, "per_sample_outputs.csv"), index=False)

print("Saved CSVs to", OUTDIR)
overall_df

eval-attack: 100%|██████████| 7/7 [02:59<00:00, 25.60s/it]
Saved CSVs to paper_outputs


,mode,mask_where,n_samples,n_fakes,auc,acc@0.5,attack_success_fake_to_real@0.5,eps,steps,eot_k
0,clean,,400,200,0.996175,0.975,NaN,NaN,NaN,NaN
1,attack,bottom,400,200,0.686900,0.595,0.795,0.031373,30.0,4.0
2,attack,top,400,200,0.727750,0.605,0.775,0.031373,30.0,4.0


In [46]:
# ============================================================
# (D) Calibration curves + ECE (clean vs attack) + save figs/CSVs
# ============================================================

# CLEAN
ece_clean, bins_clean = ece_and_bins(df_clean["y"].values, df_clean["p_fake_after"].values, n_bins=15)
bins_clean.to_csv(os.path.join(OUTDIR, "reliability_clean.csv"), index=False)
plot_reliability(bins_clean, f"Reliability (Clean) | ECE={ece_clean:.3f}", os.path.join(OUTDIR, "reliability_clean.png"))

# ATTACK bottom
ece_b, bins_b = ece_and_bins(df_attack_bottom["y"].values, df_attack_bottom["p_fake_after"].values, n_bins=15)
bins_b.to_csv(os.path.join(OUTDIR, "reliability_attack_bottom.csv"), index=False)
plot_reliability(bins_b, f"Reliability (Attack bottom) | ECE={ece_b:.3f}", os.path.join(OUTDIR, "reliability_attack_bottom.png"))

# ATTACK top
ece_t, bins_t = ece_and_bins(df_attack_top["y"].values, df_attack_top["p_fake_after"].values, n_bins=15)
bins_t.to_csv(os.path.join(OUTDIR, "reliability_attack_top.csv"), index=False)
plot_reliability(bins_t, f"Reliability (Attack top) | ECE={ece_t:.3f}", os.path.join(OUTDIR, "reliability_attack_top.png"))

# record ECEs
ece_df = pd.DataFrame([
    {"setting": "clean", "ece": ece_clean},
    {"setting": "attack_bottom", "ece": ece_b},
    {"setting": "attack_top", "ece": ece_t},
])
ece_df.to_csv(os.path.join(OUTDIR, "ece_summary.csv"), index=False)

print(ece_df)
print("Saved reliability plots + ECE CSVs to", OUTDIR)

         setting       ece
0          clean  0.167438
1  attack_bottom  0.250644
2     attack_top  0.236792
Saved reliability plots + ECE CSVs to paper_outputs


In [47]:
# ============================================================
# (E) Confidence histograms on FAKES only (before vs after) + save figs
# ============================================================

def fakes_df(df):
    return df[df["y"] == 1].copy()

f_clean = fakes_df(df_clean)
f_bottom = fakes_df(df_attack_bottom)
f_top = fakes_df(df_attack_top)

plot_conf_hist(
    f_clean["p_fake_before"].values, f_clean["p_fake_after"].values,
    "Fakes: P(fake) Clean (before vs after) [should match]",
    os.path.join(OUTDIR, "conf_hist_fakes_clean.png")
)

plot_conf_hist(
    f_bottom["p_fake_before"].values, f_bottom["p_fake_after"].values,
    "Fakes: P(fake) Attack Bottom Band (before vs after)",
    os.path.join(OUTDIR, "conf_hist_fakes_attack_bottom.png")
)

plot_conf_hist(
    f_top["p_fake_before"].values, f_top["p_fake_after"].values,
    "Fakes: P(fake) Attack Top Band (before vs after)",
    os.path.join(OUTDIR, "conf_hist_fakes_attack_top.png")
)

print("Saved confidence histograms to", OUTDIR)

Saved confidence histograms to paper_outputs


In [48]:
# ============================================================
# (F) Paper-ready per-prompt bar charts (attack success per prompt) + save figs
# ============================================================

def plot_prompt_success(per_prompt_df, title, path_png):
    d = per_prompt_df.copy()
    # Some prompt_id might be -1 if filename parsing failed; keep only >=0
    d = d[d["prompt_id"] >= 0].sort_values("prompt_id")
    plt.figure()
    plt.bar(d["prompt_id"].astype(int).values, d["attack_success@0.5"].values)
    plt.xlabel("Prompt ID")
    plt.ylabel("Attack success (fake→real @0.5)")
    plt.title(title)
    plt.savefig(path_png, dpi=200, bbox_inches="tight")
    plt.close()

plot_prompt_success(
    per_prompt_bottom,
    "Per-Prompt Attack Success (Bottom Band)",
    os.path.join(OUTDIR, "per_prompt_success_attack_bottom.png")
)

plot_prompt_success(
    per_prompt_top,
    "Per-Prompt Attack Success (Top Band)",
    os.path.join(OUTDIR, "per_prompt_success_attack_top.png")
)

print("Saved per-prompt plots to", OUTDIR)

Saved per-prompt plots to paper_outputs


In [49]:
# ============================================================
# (G) Optional: Save a few visual examples (orig vs adv) as PNG pairs
# This is great for the paper's qualitative figure.
# ============================================================
from torchvision.utils import save_image

QUAL_DIR = os.path.join(OUTDIR, "qual")
os.makedirs(QUAL_DIR, exist_ok=True)

def save_qual_examples(mask_where="bottom", n_examples=8, eps=8/255, alpha=2/255, steps=30, eot_k=4):
    model.eval()
    saved = 0
    for (x, y, prompt_id, is_fake, paths) in test_loader_meta:
        x = x.to(device); y = y.to(device); prompt_id = prompt_id.to(device)
        mask = (y == 1)
        if not mask.any():
            continue
        xf = x[mask][: (n_examples - saved)]
        pid = prompt_id[mask][: (n_examples - saved)]
        if xf.size(0) == 0:
            continue
        yt = torch.zeros(xf.size(0), dtype=torch.long, device=device)
        m = meme_band_mask(xf, band_frac=0.22, where=mask_where)
        xf_adv = eot_pgd_targeted_masked(model, xf, yt, T, eps=eps, alpha=alpha, steps=steps, eot_k=eot_k, mask=m)

        # Save images
        for i in range(xf.size(0)):
            p = int(pid[i].item())
            save_image(xf[i].detach().cpu(), os.path.join(QUAL_DIR, f"{mask_where}_p{p}_{saved}_orig.png"))
            save_image(xf_adv[i].detach().cpu(), os.path.join(QUAL_DIR, f"{mask_where}_p{p}_{saved}_adv.png"))
            saved += 1
            if saved >= n_examples:
                return

save_qual_examples("bottom", n_examples=8)
save_qual_examples("top", n_examples=8)

print("Saved qualitative examples to:", QUAL_DIR)

Saved qualitative examples to: paper_outputs/qual


In [50]:
# ============================================================
# (H) Quick "paper outputs checklist" print + file list
# ============================================================
import os

print("\n=== Paper Outputs Written ===")
for root, _, files in os.walk(OUTDIR):
    for f in sorted(files):
        if f.endswith((".csv", ".png")):
            print(os.path.join(root, f))


=== Paper Outputs Written ===
paper_outputs/conf_hist_fakes_attack_bottom.png
paper_outputs/conf_hist_fakes_attack_top.png
paper_outputs/conf_hist_fakes_clean.png
paper_outputs/ece_summary.csv
paper_outputs/overall_metrics.csv
paper_outputs/per_prompt_metrics_attack_bottom.csv
paper_outputs/per_prompt_metrics_attack_top.csv
paper_outputs/per_prompt_metrics_clean.csv
paper_outputs/per_prompt_success_attack_bottom.png
paper_outputs/per_prompt_success_attack_top.png
paper_outputs/per_sample_outputs.csv
paper_outputs/reliability_attack_bottom.csv
paper_outputs/reliability_attack_bottom.png
paper_outputs/reliability_attack_top.csv
paper_outputs/reliability_attack_top.png
paper_outputs/reliability_clean.csv
paper_outputs/reliability_clean.png
paper_outputs/qual/bottom_p0_7_adv.png
paper_outputs/qual/bottom_p0_7_orig.png
paper_outputs/qual/bottom_p2_1_adv.png
paper_outputs/qual/bottom_p2_1_orig.png
paper_outputs/qual/bottom_p2_6_adv.png
paper_outputs/qual/bottom_p2_6_orig.png
paper_outputs/q

In [8]:
from torch.utils.data import DataLoader

train_loader_meta = DataLoader(
    train_meta, batch_size=BATCH_SIZE, shuffle=True, num_workers=2,
    pin_memory=True, collate_fn=collate_meta
)
print("train_loader_meta ready:", len(train_meta))

NameError: name 'train_meta' is not defined

In [56]:
import torch
import torch.nn as nn
import torch.nn.functional as F
import torchvision.models as models
from tqdm.auto import tqdm
import numpy as np
from sklearn.metrics import roc_auc_score, accuracy_score

class ResNetDetector(nn.Module):
    def __init__(self):
        super().__init__()
        self.net = models.resnet18(weights=models.ResNet18_Weights.DEFAULT)
        self.net.fc = nn.Linear(self.net.fc.in_features, 2)
    def forward(self, x01):
        return self.net(x01)

resnet = ResNetDetector().to(device)

# Train full model but small epochs
opt_r = torch.optim.AdamW(resnet.parameters(), lr=3e-4, weight_decay=1e-4)

def eval_model(model_, loader_):
    model_.eval()
    ys, ps = [], []
    with torch.no_grad():
        for x, y, *_ in loader_:
            x = x.to(device); y = y.numpy()
            p = torch.softmax(model_(x), dim=-1)[:,1].detach().cpu().numpy()
            ys.append(y); ps.append(p)
    y = np.concatenate(ys); p = np.concatenate(ps)
    return {
        "auc": float(roc_auc_score(y, p)),
        "acc@0.5": float(accuracy_score(y, (p >= 0.5).astype(int)))
    }

# quick train (2 epochs)
for ep in range(2):
    resnet.train()
    tot = 0.0
    for x, y, *_ in tqdm(train_loader_meta, desc=f"resnet train ep{ep+1}"):
        x = x.to(device); y = y.to(device)
        opt_r.zero_grad(set_to_none=True)
        loss = F.cross_entropy(resnet(x), y)
        loss.backward()
        opt_r.step()
        tot += loss.item() * x.size(0)
    print("epoch loss:", tot / len(train_meta))

print("RESNET CLEAN:", eval_model(resnet, test_loader_meta))
torch.save(resnet.state_dict(), "paper_outputs/resnet_detector.pt")

Downloading: "https://download.pytorch.org/models/resnet18-f37072fd.pth" to /root/.cache/torch/hub/checkpoints/resnet18-f37072fd.pth
100%|██████████| 44.7M/44.7M [00:00<00:00, 208MB/s]
resnet train ep1: 100%|██████████| 25/25 [02:14<00:00,  5.38s/it]
epoch loss: 0.06274156786501407
resnet train ep2: 100%|██████████| 25/25 [01:14<00:00,  2.97s/it]
epoch loss: 0.006172823117813095
RESNET CLEAN: {'auc': 1.0, 'acc@0.5': 0.995}


In [57]:
import pandas as pd
import os
from sklearn.metrics import roc_auc_score, accuracy_score
import numpy as np
from tqdm.auto import tqdm

OUTDIR = "paper_outputs"
os.makedirs(OUTDIR, exist_ok=True)

@torch.no_grad()
def prob_fake_model(model_, x01):
    return torch.softmax(model_(x01), dim=-1)[:,1]

def eval_universal_on_model(model_, loader_meta, delta, mask_where="bottom", band_frac=0.22, use_infer_T=False, tag=""):
    model_.eval()
    rows = []
    for x, y, prompt_id, is_fake, paths in tqdm(loader_meta, desc=f"transfer-eval {tag}"):
        x = x.to(device); y = y.to(device); prompt_id = prompt_id.to(device)

        p0 = prob_fake_model(model_, x).detach()

        x_adv = x.clone()
        fm = (y == 1)
        if fm.any():
            xf = x[fm]
            m = meme_band_mask(xf, band_frac=band_frac, where=mask_where)
            xf_adv = apply_universal_delta(xf, delta.to(device), m)
            x_adv[fm] = xf_adv

        x_eval = T(x_adv) if use_infer_T else x_adv
        p1 = prob_fake_model(model_, x_eval).detach()

        for i in range(x.size(0)):
            rows.append({
                "y": int(y[i].item()),
                "prompt_id": int(prompt_id[i].item()),
                "p_fake_before": float(p0[i].item()),
                "p_fake_after": float(p1[i].item()),
                "mask_where": mask_where,
                "use_infer_T": int(use_infer_T),
                "model": tag
            })
    df = pd.DataFrame(rows)

    y_true = df["y"].values
    p = df["p_fake_after"].values
    auc = roc_auc_score(y_true, p)
    acc = accuracy_score(y_true, (p >= 0.5).astype(int))

    df_f = df[df["y"] == 1]
    succ = float((df_f["p_fake_after"].values < 0.5).mean())

    overall = {
        "model": tag,
        "mask_where": mask_where,
        "use_infer_T": int(use_infer_T),
        "auc_after": float(auc),
        "acc@0.5_after": float(acc),
        "attack_success_fake_to_real@0.5": float(succ),
        "n_fakes": int((df["y"]==1).sum()),
        "n_total": int(len(df)),
    }
    return df, overall

In [66]:
# ============================================================
# UNIVERSAL MASKED EOT ATTACK (TRAIN ONE δ) + PAPER-READY OUTPUTS
# Assumptions:
#  - model(x01) -> logits for [real,fake] given x01 in [0,1]
#  - T is your FBLikeTransforms() module (differentiable)
#  - meme_band_mask(x, ...) exists
#  - test_loader_meta exists (x,y,prompt_id,is_fake,paths)
# ============================================================

import os, math
import numpy as np
import pandas as pd
import torch
import torch.nn.functional as F
import matplotlib.pyplot as plt
from tqdm.auto import tqdm
from sklearn.metrics import roc_auc_score, accuracy_score

OUTDIR = "paper_outputs"
os.makedirs(OUTDIR, exist_ok=True)

@torch.no_grad()
def prob_fake01(x01):
    model.eval()
    return torch.softmax(model(x01), dim=-1)[:, 1]

def apply_universal_delta(x01, delta, mask):
    # delta and mask broadcastable to x01
    return (x01 + delta * mask).clamp(0, 1)

print("Ready ✅")

Ready ✅


In [78]:
import torch
import torch.nn.functional as F
from tqdm.auto import tqdm

def train_universal_masked_eot_stable(
    model,
    train_loader_meta,
    T,
    mask_where="bottom",
    band_frac=0.22,
    eps_u=8/255,
    step_u=2/255,          # PGD-like step size
    iters=300,
    eot_k=4,
    device="cuda"
):
    """
    Stable universal delta training:
      δ <- Proj_{||δ||∞<=eps} ( δ - step * sign(grad) )
    with NaN guards + enforced masking.
    """
    model.eval()
    delta = torch.zeros(1, 3, 224, 224, device=device)

    loader_iter = iter(train_loader_meta)

    for it in tqdm(range(iters), desc=f"train-universal-stable-{mask_where}"):
        try:
            x, y, *_ = next(loader_iter)
        except StopIteration:
            loader_iter = iter(train_loader_meta)
            x, y, *_ = next(loader_iter)

        x = x.to(device); y = y.to(device)
        fm = (y == 1)
        if fm.sum() == 0:
            continue
        xf = x[fm]

        # mask (same for all, take first)
        m = meme_band_mask(xf, band_frac=band_frac, where=mask_where)
        m1 = m[:1].detach()

        target = torch.zeros(xf.size(0), dtype=torch.long, device=device)

        # compute grad wrt delta
        delta_var = delta.clone().detach().requires_grad_(True)

        loss = 0.0
        for _ in range(eot_k):
            adv = (xf + delta_var * m).clamp(0,1)
            adv_t = T(adv)
            logits = model(adv_t)
            loss = loss + F.cross_entropy(logits, target)
        loss = loss / eot_k

        grad = torch.autograd.grad(loss, delta_var, retain_graph=False, create_graph=False)[0]

        # NaN guard
        grad = torch.nan_to_num(grad, nan=0.0, posinf=0.0, neginf=0.0)

        # targeted: descend loss
        with torch.no_grad():
            delta = delta - step_u * grad.sign()
            delta = torch.clamp(delta, -eps_u, eps_u)
            delta = delta * m1  # enforce only in band
            delta = torch.nan_to_num(delta, nan=0.0, posinf=0.0, neginf=0.0)

    return delta.detach()

# Re-train universal deltas (use train_loader_meta ideally)
delta_u_bottom = train_universal_masked_eot_stable(
    model, train_loader_meta, T, mask_where="bottom",
    eps_u=8/255, step_u=2/255, iters=300, eot_k=4, device=device
)
delta_u_top = train_universal_masked_eot_stable(
    model, train_loader_meta, T, mask_where="top",
    eps_u=8/255, step_u=2/255, iters=300, eot_k=4, device=device
)

# Save
import os
OUTDIR="paper_outputs"; os.makedirs(OUTDIR, exist_ok=True)
torch.save(delta_u_bottom.cpu(), os.path.join(OUTDIR, "universal_delta_bottom.pt"))
torch.save(delta_u_top.cpu(), os.path.join(OUTDIR, "universal_delta_top.pt"))

print("Universal deltas retrained + saved ✅")

train-universal-stable-top: 100%|██████████| 300/300 [13:03<00:00,  2.61s/it]
Universal deltas retrained + saved ✅


In [79]:
import numpy as np
import pandas as pd
from sklearn.metrics import roc_auc_score, accuracy_score
from tqdm.auto import tqdm

@torch.no_grad()
def prob_fake_model(model_, x01):
    p = torch.softmax(model_(x01), dim=-1)[:,1]
    # sanitize in torch
    p = torch.nan_to_num(p, nan=0.5, posinf=1.0, neginf=0.0)
    return p

def eval_universal_delta_safe(
    model_, loader_meta, delta, mask_where="bottom", band_frac=0.22,
    use_infer_T=False, tag="universal"
):
    model_.eval()
    rows = []

    for x, y, prompt_id, is_fake, paths in tqdm(loader_meta, desc=f"eval-{tag}-{mask_where}"):
        x = x.to(device); y = y.to(device); prompt_id = prompt_id.to(device)

        p0 = prob_fake_model(model_, x).detach()

        x_adv = x.clone()
        fm = (y == 1)
        if fm.any():
            xf = x[fm]
            m = meme_band_mask(xf, band_frac=band_frac, where=mask_where)
            xf_adv = (xf + delta.to(device) * m).clamp(0,1)
            x_adv[fm] = xf_adv

        x_eval = T(x_adv) if use_infer_T else x_adv
        p1 = prob_fake_model(model_, x_eval).detach()

        for i in range(x.size(0)):
            rows.append({
                "y": int(y[i].item()),
                "prompt_id": int(prompt_id[i].item()),
                "p_fake_before": float(p0[i].item()),
                "p_fake_after": float(p1[i].item()),
                "mask_where": mask_where,
                "use_infer_T": int(use_infer_T),
                "model": tag
            })

    df = pd.DataFrame(rows)

    # sanitize numpy arrays too
    y_true = df["y"].to_numpy().astype(int)
    p = df["p_fake_after"].to_numpy().astype(float)
    bad = np.isnan(p).sum() + np.isinf(p).sum()
    if bad > 0:
        print(f"WARNING: Found {bad} NaN/Inf probs. Replacing with 0.5 for metrics.")
        p = np.nan_to_num(p, nan=0.5, posinf=1.0, neginf=0.0)
        df["p_fake_after"] = p

    auc = roc_auc_score(y_true, p)
    acc = accuracy_score(y_true, (p >= 0.5).astype(int))

    df_f = df[df["y"] == 1]
    succ = float((df_f["p_fake_after"].to_numpy() < 0.5).mean())

    overall = {
        "model": tag,
        "mask_where": mask_where,
        "use_infer_T": int(use_infer_T),
        "auc_after": float(auc),
        "acc@0.5_after": float(acc),
        "attack_success_fake_to_real@0.5": float(succ),
        "n_total": int(len(df)),
        "n_fakes": int((df["y"]==1).sum()),
    }
    return df, overall

# Run evals again
df_u_bot, overall_u_bot = eval_universal_delta_safe(model, test_loader_meta, delta_u_bottom, "bottom", use_infer_T=False, tag="CLIP")
df_u_bot_T, overall_u_bot_T = eval_universal_delta_safe(model, test_loader_meta, delta_u_bottom, "bottom", use_infer_T=True, tag="CLIP")

import pandas as pd, os
pd.DataFrame([overall_u_bot, overall_u_bot_T]).to_csv(os.path.join(OUTDIR, "universal_overall_metrics_clip_bottom.csv"), index=False)
print(overall_u_bot)
print(overall_u_bot_T)

eval-CLIP-bottom: 100%|██████████| 7/7 [00:27<00:00,  3.94s/it]
{'model': 'CLIP', 'mask_where': 'bottom', 'use_infer_T': 0, 'auc_after': 0.9398250000000001, 'acc@0.5_after': 0.885, 'attack_success_fake_to_real@0.5': 0.215, 'n_total': 400, 'n_fakes': 200}
{'model': 'CLIP', 'mask_where': 'bottom', 'use_infer_T': 1, 'auc_after': 0.9494, 'acc@0.5_after': 0.8725, 'attack_success_fake_to_real@0.5': 0.245, 'n_total': 400, 'n_fakes': 200}


In [1]:
# delta_u_bottom and delta_u_top from your universal training
transfer_rows = []

# Evaluate on CLIP model
df_clip_u, overall_clip_u = eval_universal_on_model(model, test_loader_meta, delta_u_bottom, "bottom", tag="CLIP", use_infer_T=False)
df_clip_u_T, overall_clip_u_T = eval_universal_on_model(model, test_loader_meta, delta_u_bottom, "bottom", tag="CLIP", use_infer_T=True)

# Evaluate on ResNet model
df_res_u, overall_res_u = eval_universal_on_model(resnet, test_loader_meta, delta_u_bottom, "bottom", tag="ResNet18", use_infer_T=False)
df_res_u_T, overall_res_u_T = eval_universal_on_model(resnet, test_loader_meta, delta_u_bottom, "bottom", tag="ResNet18", use_infer_T=True)

transfer_overall = pd.DataFrame([overall_clip_u, overall_clip_u_T, overall_res_u, overall_res_u_T])
transfer_overall.to_csv("paper_outputs/transfer_universal_bottom.csv", index=False)

transfer_overall

NameError: name 'eval_universal_on_model' is not defined

<a style='text-decoration:none;line-height:16px;display:flex;color:#5B5B62;padding:10px;justify-content:end;' href='https://deepnote.com?utm_source=created-in-deepnote-cell&projectId=dbc8aaad-7458-4d3a-af75-0f37995653fb' target="_blank">

Created in <span style='font-weight:600;margin-left:4px;'>Deepnote</span></a>